# B. Лаборатория биокибернетики [csv файл]

Исследователи из лаборатории биокибернетики разработали алгоритм для классификации биологических объектов.

Однажды младший научный сотрудник предложил простое определение: «Объект класса X — это организм с двумя конечностями и без перьевого покрова». Но старший коллега, известный своим остроумием, принёс в лабораторию ощипанную курицу и заявил: «По вашему определению, это тоже объект X!». Пришлось уточнить критерий: «…и с плоскими когтями».

Чтобы автоматизировать классификацию, исследователи закодировали признаки организмов латинскими буквами от A до I. Они не раскрыли, что означает каждый признак, поэтому их интерпретация остаётся неизвестной.

Позже команда собрала данные:

Младший сотрудник подготовил обучающую выборку с метками.
Старший — тестовую выборку, но забыл указать классы объектов и ушёл на перерыв.
Задача: Используя обучающие данные, предскажите метки классов для тестовой выборки.

## О датасете

Вам предлагаются три файла, `train.csv`, `test.csv` и `example.csv`:

- Файл `train.csv` содержит признаки обучающей выборки и колонку с разметкой `target`. Значение 1 в этой колонке соответствует человеку, а 0 - нечеловеку.
- Файл `test.csv` содержит признаки тестовой выборки.
- Файл `example.csv` содержит пример корректной посылки в контест.
Таким образом, вам нужно предсказать колонку target для объектов из файла `test.csv`.

## Что нужно сделать

От вас требуется загрузить в систему файл `answers.csv` в формате, аналогичном файлу `example.csv` с предсказаниями для объектов тестовой выборки. В качестве целевой метрики используется ROC-AUC. Баллы за это задание рассчитываются по формуле:

`100*max(min((AUC−0.8)/0.08,1),0)`,

где AUC - значение ROC-AUC ваших предсказаний. Таким образом, для максимального числа баллов необходимо набрать `AUC≥0.88`.

## Ответ
Язык
(make) yandexdataschool

In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv('train.csv')
train_df['C'] = train_df['C'].map({'-': 0, '+': 1})
train_df

,A,B,C,D,E,F,G,H,I,target
0,0.505,8,0,1.984,3.0,5,2.642,-5.122,0.649,1
1,0.536,4,0,1.977,1.0,3,5.756,-3.077,0.950,0
2,0.024,3,0,3.147,2.0,6,2.435,4.387,2.186,1
3,0.543,4,0,2.440,3.0,9,4.440,7.730,1.938,0
4,0.942,8,0,1.952,3.0,9,7.176,-4.579,0.346,1
...,...,...,...,...,...,...,...,...,...,...
995,0.985,3,0,2.233,1.0,8,5.876,6.461,1.836,0
996,0.369,5,0,1.947,1.0,3,4.922,3.857,2.213,0
997,0.454,11,0,2.184,1.0,3,7.198,9.365,2.587,0
998,0.012,7,0,1.832,1.0,8,5.290,3.528,2.364,0


In [3]:
train_df.corr()

,A,B,C,D,E,F,G,H,I,target
A,1.000000,0.039094,0.022909,0.050887,0.004346,0.007045,0.004258,-0.000557,-0.398194,-0.114122
B,0.039094,1.000000,-0.054097,0.001675,-0.015530,0.002693,0.013723,0.032052,0.015567,-0.027785
C,0.022909,-0.054097,1.000000,-0.067846,-0.038455,-0.036167,-0.000594,0.020045,-0.003402,-0.004121
D,0.050887,0.001675,-0.067846,1.000000,0.042754,-0.041497,0.000281,-0.041080,-0.025851,0.119772
E,0.004346,-0.015530,-0.038455,0.042754,1.000000,0.016497,0.010080,-0.016537,-0.023114,0.397967
F,0.007045,0.002693,-0.036167,-0.041497,0.016497,1.000000,0.025060,-0.007663,-0.020457,0.010394
G,0.004258,0.013723,-0.000594,0.000281,0.010080,0.025060,1.000000,-0.005418,-0.010506,0.126767
H,-0.000557,0.032052,0.020045,-0.041080,-0.016537,-0.007663,-0.005418,1.000000,0.813983,-0.245640
I,-0.398194,0.015567,-0.003402,-0.025851,-0.023114,-0.020457,-0.010506,0.813983,1.000000,-0.156417
target,-0.114122,-0.027785,-0.004121,0.119772,0.397967,0.010394,0.126767,-0.245640,-0.156417,1.000000


In [4]:
train_df.isna().sum()

A          0
B          0
C          0
D          0
E         75
F          0
G          0
H          0
I          0
target     0
dtype: int64

In [5]:
train_df['E'].value_counts()

E
3.0    333
1.0    299
2.0    293
Name: count, dtype: int64

In [6]:
test_df = pd.read_csv('test.csv')
test_df['C'] = test_df['C'].map({'-': 0, '+': 1})
test_df

,A,B,C,D,E,F,G,H,I
0,0.489,4,1,1.722,3.0,2,5.727,3.126,1.858
1,0.479,8,0,2.282,3.0,8,4.955,2.650,2.769
2,0.640,2,1,1.918,2.0,2,5.971,1.216,2.062
3,0.649,6,0,1.696,2.0,2,4.946,-0.480,1.133
4,0.272,1,0,1.721,1.0,7,4.670,8.785,5.090
...,...,...,...,...,...,...,...,...,...
495,0.273,1,0,2.006,NaN,10,4.722,8.541,5.063
496,0.996,3,0,1.743,1.0,1,5.415,-10.913,-0.169
497,0.550,5,0,1.770,3.0,3,3.998,5.320,2.054
498,0.966,2,1,1.984,1.0,6,5.999,-5.422,0.534


In [7]:
test_df.isna().sum()

A     0
B     0
C     0
D     0
E    43
F     0
G     0
H     0
I     0
dtype: int64

In [8]:
test_df['E'].value_counts()

E
2.0    162
3.0    156
1.0    139
Name: count, dtype: int64

# KNN

In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_curve, auc

In [10]:
X = train_df[['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']].fillna(0)
y = train_df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scale features (important for distance-based algorithms like KNN)
scaler = MinMaxScaler()  #StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the KNN classifier
# n_neighbors specifies the 'k' value
knn = KNeighborsClassifier(n_neighbors=27, weights='distance', metric='manhattan')
#knn = KNeighborsClassifier(n_neighbors=13, weights='distance', metric='euclidean')
knn.fit(X_train_scaled, y_train)

,n_neighbors,27
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'manhattan'
,metric_params,None
,n_jobs,None


In [11]:
y_pred_1 = knn.predict_proba(X_test_scaled)[:, 1] # Probabilities for the positive class

fpr, tpr, thresholds = roc_curve(y_test, y_pred_1)
roc_auc = auc(fpr, tpr)
print(roc_auc)
print(100 * max(min((roc_auc - 0.8) / 0.08, 1), 0))

0.8394024276377218
49.25303454715216


In [ ]:
k_range = list(range(1, 33))
weight_options = ['uniform', 'distance']
metric_options = ['euclidean', 'manhattan', 'minkowski']

param_grid = dict(n_neighbors=k_range, weights=weight_options, metric=metric_options)
grid = GridSearchCV(knn, param_grid, cv=10, scoring='accuracy', return_train_score=False)

scaler = MinMaxScaler()  # StandardScaler()
X_scaled = scaler.fit_transform(X)
grid.fit(X_scaled, y)

,estimator,KNeighborsCla...ts='distance')
,param_grid,"{'metric': ['euclidean', 'manhattan', ...], 'n_neighbors': [1, 2, ...], 'weights': ['uniform', 'distance']}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,17


In [13]:
grid_mean_scores = grid.cv_results_['mean_test_score']
print(grid_mean_scores)

[0.744 0.744 0.757 0.744 0.769 0.769 0.771 0.774 0.776 0.775 0.776 0.772
 0.773 0.773 0.767 0.781 0.773 0.777 0.782 0.782 0.781 0.781 0.778 0.778
 0.782 0.786 0.776 0.781 0.789 0.79  0.769 0.786 0.779 0.782 0.766 0.774
 0.77  0.776 0.767 0.772 0.766 0.769 0.772 0.768 0.766 0.775 0.766 0.773
 0.768 0.773 0.771 0.774 0.767 0.773 0.77  0.772 0.765 0.771 0.77  0.768
 0.766 0.769 0.768 0.765 0.745 0.745 0.758 0.745 0.775 0.772 0.778 0.77
 0.773 0.773 0.769 0.775 0.772 0.777 0.773 0.783 0.775 0.78  0.773 0.779
 0.786 0.792 0.774 0.781 0.78  0.788 0.775 0.779 0.789 0.792 0.778 0.782
 0.789 0.795 0.786 0.789 0.787 0.789 0.78  0.788 0.79  0.789 0.778 0.785
 0.782 0.787 0.779 0.783 0.777 0.785 0.776 0.781 0.778 0.785 0.777 0.781
 0.773 0.779 0.773 0.782 0.772 0.776 0.776 0.779 0.744 0.744 0.757 0.744
 0.769 0.769 0.771 0.774 0.776 0.775 0.776 0.772 0.773 0.773 0.767 0.781
 0.773 0.777 0.782 0.782 0.781 0.781 0.778 0.778 0.782 0.786 0.776 0.781
 0.789 0.79  0.769 0.786 0.779 0.782 0.766 0.774 0.7

In [14]:
print(grid.best_score_)
print(grid.best_params_)

0.795
{'metric': 'manhattan', 'n_neighbors': 17, 'weights': 'distance'}


In [15]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
knn_res = KNeighborsClassifier(n_neighbors=27, weights='distance', metric='manhattan')
knn_res.fit(X_scaled, y)
# pred_res = knn_res.predict(scaler.transform(test_df.fillna(0)))
pred_res = knn_res.predict_proba(scaler.transform(test_df.fillna(0)))[:, 1]

pd.DataFrame({'target': pred_res}).to_csv('result.csv', index=False)

# Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

In [17]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

le = LabelEncoder()
train_df['C'] = le.fit_transform(train_df['C'])
test_df['C'] = le.transform(test_df['C'])

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, criterion='entropy', random_state=42)
model.fit(X_train_imputed, y_train)

y_pred = model.predict_proba(X_test_imputed)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)
print(roc_auc)
print(100 * max(min((roc_auc - 0.8) / 0.08, 1), 0))

0.8576945929887106
72.11824123588825


In [19]:
imputer = SimpleImputer(strategy='median')
X_train = train_df.drop('target', axis=1)
y_train = train_df['target']
X_test = test_df

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, criterion='entropy', random_state=42)
model.fit(X_train_imputed, y_train)

y_pred = model.predict_proba(X_test_imputed)[:, 1]

result = pd.DataFrame({'target': y_pred})
result.to_csv('result.csv', index=False)

# CatBoost

In [20]:
from catboost import CatBoostClassifier

In [21]:
train_df['target'].value_counts()

target
0    712
1    288
Name: count, dtype: int64

In [22]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

categorical_features = ['C']

X = train_df.drop('target', axis=1)
y = train_df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


# CatBoost может сам обрабатывать категориальные признаки и пропущенные значения
model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.1,
    depth=6,
    cat_features=categorical_features,
    #auto_class_weights='Balanced',
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    eval_metric='AUC',  # Используем AUC для валидации
    use_best_model=True,
    # od_type="Iter",
    # od_wait=100
)

model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=100)

y_pred = model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)

print(roc_auc)
print(100 * max(min((roc_auc - 0.8) / 0.08, 1), 0))

0:	test: 0.7912682	best: 0.7912682 (0)	total: 68.5ms	remaining: 1m 42s
100:	test: 0.8810041	best: 0.8821452 (71)	total: 141ms	remaining: 1.95s
200:	test: 0.8887742	best: 0.8892089 (198)	total: 216ms	remaining: 1.39s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8892088676
bestIteration = 198

Shrink model to first 199 iterations.
0.8892088676374701
100


In [23]:
model = CatBoostClassifier(
    iterations=198,
    learning_rate=0.1,
    depth=6,
    cat_features=categorical_features,
    #auto_class_weights='Balanced',
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    eval_metric='AUC',  # Используем AUC для валидации
)

model.fit(X, y, verbose=100)
y_pred = model.predict_proba(test_df)[:, 1]
result = pd.DataFrame({'target': y_pred})
result.to_csv('result.csv', index=False)

0:	total: 1.51ms	remaining: 298ms
100:	total: 67ms	remaining: 64.3ms
197:	total: 120ms	remaining: 0us
